# dim_player.py

## Purpose
Builds `dim_player` as an SCD2 dimension that tracks descriptive player attributes.

## Sources
- Bronze: `fifa.bronze.ea_fc_players_raw`
- Silver: `dim_gender`, `dim_nationality`, `dim_club`, `dim_position`, `dim_edition`

## Business key
- `player_id` + `edition_id` (natural key from EA, stable across snapshots)

## Strategy
- SCD2 with MERGE
- Metrics (34 detailed attributes) go to `fact_player_snapshot`, NOT here
- Only descriptive and slowly-changing attributes in this dimension
- Tracked attributes: `club_id`, `position_id`, `overall_rating`, `alternate_positions`, `playstyles`, `preferred_foot`, `skill_moves`, `weak_foot`

## Output
- Table: `fifa.silver.dim_player`

## Columns
- `player_snapshot_key` (STRING): composite key = `player_id + snapshot_date`
- `player_id` (INT)
- `edition_id` (INT): FK to `dim_edition`
- Names: `first_name`, `last_name`, `common_name`
- FKs: `gender_id`, `nationality_id`, `club_id`, `position_id`
- Descriptive: `birthdate`, `preferred_foot`, `alternate_positions`, `playstyles`
- Card attributes: `overall_rating`, `skill_moves`, `weak_foot`
- SCD2 control: `Start_date`, `End_date`, `Is_current`
- Metadata: `_ingestion_timestamp`

## Why SCD2 with `player_id + edition_id`?

Each edition has its own current version of a player. For example:

- FC 26: Mbappé at PSG (`Is_current = true` for edition 2026)
- FC 27: Mbappé at Real Madrid (`Is_current = true` for edition 2027)

Using `player_id + edition_id` as the business key ensures that each edition has its own current version, instead of overwriting the previous one.

## Why `player_snapshot_key`?

`player_id` alone is not unique in `fact_player_snapshot` (a player has one row per snapshot). The composite key `player_id + snapshot_date` solves two problems:

1. **Power BI relationships**: it gives a unique key for the relationship between `dim_player` and `fact_player_snapshot`.
2. **SCD2 correctness**: each player-version pair is uniquely identifiable.

## Metrics location

The 34 detailed attributes (pace, shooting, passing, etc.) do NOT go in `dim_player`. They go in `fact_player_snapshot`, because:

- They are measures, not descriptors.
- They change with every snapshot (patch updates).
- Putting them in the dimension would duplicate them across SCD2 versions.

In [0]:
from pyspark.sql.functions import col, current_timestamp, to_date, lit, regexp_extract, when, concat,trim, explode, split, pmod, xxhash64

# ===== Read Bronze =====
bronze = spark.read.table("fifa.bronze.ea_fc_players_raw")

# 
players = (
    bronze
    .select(
        col("player_id").cast("int"),
        col("edition").alias("edition_code"),
        trim(col("first_name")).alias("first_name"),
        trim(col("last_name")).alias("last_name"),
        trim(col("common_name")).alias("common_name"),
        trim(col("gender")).alias("gender"),
        trim(col("league")).alias("league"),
        trim(col("nationality")).alias("nationality"),
        to_date(col("birthdate"), "yyyy-MM-dd").alias("birthdate"),
        trim(col("preferred_foot")).alias("preferred_foot"),
        trim(col("club")).alias("club"),
        trim(col("position")).alias("position"),
        when(col("alternate_positions").isNull(), lit(""))
            .otherwise(trim(col("alternate_positions"))).alias("alternate_positions"),
        when(col("playstyles").isNull(), lit(""))
            .otherwise(trim(col("playstyles"))).alias("playstyles"),
        col("overall_rating").cast("int").alias("overall_rating"),
        col("skill_moves").cast("int").alias("skill_moves"),
        col("weak_foot").cast("int").alias("weak_foot"),
        col("snapshot_date").alias("snapshot_date"),
    )
    .distinct()
)

# ===== Load dimensions =====
dim_edition = spark.read.table("fifa.silver.dim_edition").alias("e")
dim_gender = spark.read.table("fifa.silver.dim_gender").alias("g")
dim_league = spark.read.table("fifa.silver.dim_league").alias("l")
dim_nationality = spark.read.table("fifa.silver.dim_nationality").alias("n")
dim_club = spark.read.table("fifa.silver.dim_club").filter(col("Is_current") == True).alias("c")
dim_position = spark.read.table("fifa.silver.dim_position").alias("p")

# ===== Enrich players with dimension FKs =====
players_enriched = (
    players.alias("pl")
    .join(dim_gender, col("pl.gender") == col("g.gender_name"), how="left")
    .join(dim_league, col("pl.league") == col("l.league_name"), how="left")
    .join(dim_nationality, col("pl.nationality") == col("n.nationality_name"), how="left")
    .join(dim_club, 
          (col("pl.club") == col("c.club_name")) & (col("g.gender_id") == col("c.gender_id")) & (col("l.league_id") == col("c.league_id")),
          how="left")
    .join(dim_position, col("pl.position") == col("p.position_code"), how="left")
    .join(dim_edition, col("pl.edition_code") == col("e.edition_code"), how="left")
    .select(
        col("pl.*"),
        col("g.gender_id").alias("gender_id"),
        col("n.nationality_id").alias("nationality_id"),
        col("c.club_id").alias("club_id"),
        col("p.position_id").alias("position_id"),
        col("e.edition_id").alias("edition_id") 
    )
)

# ===== Build SCD2-ready DataFrame =====
new_players = (
    players_enriched
    .select(
        "player_id", "edition_id","first_name", "last_name", "common_name",
        "gender_id", "nationality_id", "birthdate", "preferred_foot",
        "club_id", "position_id", "alternate_positions", "playstyles",
        "overall_rating", "skill_moves", "weak_foot", "snapshot_date"
    )
     # Composite key: player_id + snapshot_date
    .withColumn("player_snapshot_key", concat(col("player_id"), lit("-"), col("snapshot_date"))) 
    .withColumn("Start_date", current_timestamp())
    .withColumn("End_date", lit(None).cast("timestamp"))
    .withColumn("Is_current", lit(True))
    .withColumn("_ingestion_timestamp", current_timestamp())
)


# ===== Create table Silver =====
spark.sql("""
CREATE TABLE IF NOT EXISTS fifa.silver.dim_player (
    player_snapshot_key STRING,
    player_id INT,
    edition_id INT,  
    first_name STRING,
    last_name STRING,
    common_name STRING,
    gender_id INT,
    nationality_id INT,
    birthdate DATE,
    preferred_foot STRING,
    club_id INT,
    position_id INT,
    alternate_positions STRING,
    playstyles STRING,
    overall_rating INT,
    skill_moves INT,
    weak_foot INT,
    snapshot_date DATE,  
    Start_date TIMESTAMP,
    End_date TIMESTAMP,
    Is_current BOOLEAN,
    _ingestion_timestamp TIMESTAMP
)
USING DELTA;
""")

# MERGE SCD2
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "fifa.silver.dim_player")

(
    target.alias("t")
    .merge(
        new_players.alias("s"),
        "t.player_id = s.player_id AND t.edition_id = s.edition_id AND t.Is_current = true"
    )
    # Expire the current record if the snapshot is more recent and any tracked attribute changed
    .whenMatchedUpdate(
        condition="""
        s.snapshot_date > t.snapshot_date AND (
            t.club_id <> s.club_id OR
            t.position_id <> s.position_id OR
            t.overall_rating <> s.overall_rating OR
            t.alternate_positions <> s.alternate_positions OR
            t.playstyles <> s.playstyles OR
            t.preferred_foot <> s.preferred_foot OR
            t.skill_moves <> s.skill_moves OR
            t.weak_foot <> s.weak_foot )
        """,
        set={
            "End_date": "s.Start_date",
            "Is_current": "false"
        }
    )
    .whenNotMatchedInsert(
        values={
            "player_snapshot_key": "s.player_snapshot_key",  
            "player_id": "s.player_id",
            "edition_id": "s.edition_id",
            "first_name": "s.first_name",
            "last_name": "s.last_name",
            "common_name": "s.common_name",
            "gender_id": "s.gender_id",
            "nationality_id": "s.nationality_id",
            "birthdate": "s.birthdate",
            "preferred_foot": "s.preferred_foot",
            "club_id": "s.club_id",
            "position_id": "s.position_id",
            "alternate_positions": "s.alternate_positions",
            "playstyles": "s.playstyles",
            "overall_rating": "s.overall_rating",
            "skill_moves": "s.skill_moves",
            "weak_foot": "s.weak_foot",
            "snapshot_date": "s.snapshot_date",  
            "Start_date": "s.Start_date",
            "End_date": "s.End_date",
            "Is_current": "s.Is_current",
            "_ingestion_timestamp": "s._ingestion_timestamp"
        }
    )
    .execute()
)

print(f"dim_player actualizada con {spark.read.table('fifa.silver.dim_player').count()} filas")

LIMIT 10 of the dim_player table. Quick verification that we have entered the raw data into our table.

In [0]:
%sql
SELECT * FROM fifa.silver.dim_player ORDER BY overall_rating DESC limit 10;